<div align="center">

### MySQL & SQL Analysis

**Loading the cleaned product data into MySQL and performing business analysis using SQL queries.**

</div>

<div align="center">

### Workflow

**Cleaned CSV → MySQL Database → Create Table → Import Data → Verify Data → SQL Queries → Business Analysis → Insights**

</div>

<div align="center">

###  Libraries & Tools

**Tools used for database management and SQL analysis.**



</div>

In [1]:
%pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


<div align="center">

###  1. Load Data into Database

**Loading the cleaned CSV dataset into an SQLite database and creating the `products` table for SQL analysis.**

</div>

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

# Find CSV
notebook_dir = Path.cwd()

csv_candidates = [
    notebook_dir / "Data/shopsy_db/shopsy_kitchen_products_cleaned.csv",
    notebook_dir.parent / "Data/shopsy_db/shopsy_kitchen_products_cleaned.csv",
    Path("Data/shopsy_db/shopsy_kitchen_products_cleaned.csv")
]

csv_path = next((path for path in csv_candidates if path.exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "shopsy_kitchen_products_cleaned.csv not found."
    )

# Load CSV
df = pd.read_csv(csv_path)

# Clean column names
df.columns = df.columns.str.lower().str.strip()

print("CSV loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Create SQLite database
query_conn = sqlite3.connect(":memory:")

# Load dataframe into SQLite
df.to_sql(
    "products",
    query_conn,
    index=False,
    if_exists="replace"
)

print("Products table created successfully")

CSV loaded successfully
Shape: (29, 9)
Columns: ['product_name', 'price', 'discount', 'rating', 'reviews', 'brand', 'capacity', 'material', 'product_url']
Products table created successfully


<div align="center">

###  2. Top 10 Most Reviewed Products

**Identifying the products with the highest number of customer reviews.**

</div>

In [3]:
query = """
SELECT product_name, brand, price, rating, reviews
FROM products
WHERE reviews IS NOT NULL
ORDER BY reviews DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,product_name,brand,price,rating,reviews
0,koktailkitchan Pack of 12 Plastic Grocery Cont...,koktailkitchan,322,4.1,314.0
1,BELIZZI Pack of 6 Plastic Fridge Container - 1...,BELIZZI,257,4.1,272.0
2,VR Pack of 3 Plastic Grocery Container - 4500 ...,VR,247,4.1,210.0
3,Qtrix Pack of 8 Plastic Grocery Container - 50...,Qtrix,300,4.4,104.0
4,GRCOXZ Pack of 12 Plastic Grocery Container - ...,GRCOXZ,560,4.1,94.0
5,"RK Pack of 5 Steel Cookie Jar - 300 ml, 500 ml...",RK,383,4.1,83.0
6,VR Pack of 4 Plastic Fridge Container - 2500 m...,VR,292,4.1,70.0
7,SKYHEART Pack of 8 Plastic Grocery Container -...,SKYHEART,486,4.4,61.0
8,VAD Pack of 2 Glass Grocery Container - 1000 m...,VAD,169,4.3,43.0
9,AneriDEALS Pack of 24 Plastic Grocery Containe...,AneriDEALS,423,4.1,34.0


<div align="center">

### 3. Top-Rated Products

**Identifying the highest-rated products and ranking them by customer reviews.**

</div>

In [4]:
query = """
SELECT product_name, brand, price, rating, reviews
FROM products
WHERE rating IS NOT NULL
  AND reviews IS NOT NULL
ORDER BY rating DESC, reviews DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,product_name,brand,price,rating,reviews
0,Qtrix Pack of 8 Plastic Grocery Container - 50...,Qtrix,300,4.4,104.0
1,SKYHEART Pack of 8 Plastic Grocery Container -...,SKYHEART,486,4.4,61.0
2,Flatkitch Pack of 6 Plastic Grocery Container ...,Flatkitch,241,4.4,1.0
3,VAD Pack of 2 Glass Grocery Container - 1000 m...,VAD,169,4.3,43.0
4,Veksin Pack of 6 Plastic Grocery Container - 1...,Veksin,497,4.3,23.0
5,MegaValue Pack of 6 Plastic Grocery Container ...,MegaValue,242,4.3,11.0
6,INNOVIX Pack of 1 Plastic Cereal Dispenser - 2...,INNOVIX,175,4.3,8.0
7,Hoatzin Pack of 6 Plastic Grocery Container - ...,Hoatzin,497,4.3,7.0
8,Q7 BRAND Pack of 3 Plastic Grocery Container -...,Q7 BRAND,363,4.2,4.0
9,INNOVIX Pack of 2 Plastic Cereal Dispenser - 2...,INNOVIX,283,4.2,4.0


<div align="center">

###  4. Brand Performance Analysis

**Analyzing the top brands based on product count, average price, average rating, and total reviews.**

</div>

In [5]:
query = """
SELECT 
    brand,
    COUNT(*) AS product_count,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(rating), 2) AS average_rating,
    COALESCE(SUM(reviews), 0) AS total_reviews
FROM products
WHERE brand IS NOT NULL
GROUP BY brand
ORDER BY total_reviews DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,brand,product_count,average_price,average_rating,total_reviews
0,koktailkitchan,1,322.00,4.10,314.0
1,VR,3,270.67,4.13,283.0
2,BELIZZI,1,257.00,4.10,272.0
3,Qtrix,1,300.00,4.40,104.0
4,GRCOXZ,1,560.00,4.10,94.0
5,RK,2,358.00,4.10,86.0
6,SKYHEART,1,486.00,4.40,61.0
7,VAD,1,169.00,4.30,43.0
8,AneriDEALS,1,423.00,4.10,34.0
9,VASOYA,1,155.00,4.00,25.0


<div align="center">

###  5. Discount Analysis

**Analyzing average, highest, lowest, and available product discounts.**

</div>

In [6]:
query = """
SELECT 
    ROUND(AVG(discount), 2) AS average_discount,
    MAX(discount) AS highest_discount,
    MIN(discount) AS lowest_discount,
    COUNT(*) AS products_with_discount
FROM products
WHERE discount IS NOT NULL
"""

display(pd.read_sql(query, query_conn))

,average_discount,highest_discount,lowest_discount,products_with_discount
0,61.62,86,6,29


<div align="center">

###  6. Brand-wise Price Analysis

**Analyzing minimum, average, and maximum product prices across brands.**

</div>

In [7]:
query = """
SELECT 
    brand,
    COUNT(*) AS product_count,
    ROUND(MIN(price), 2) AS minimum_price,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(MAX(price), 2) AS maximum_price
FROM products
WHERE brand IS NOT NULL
  AND price IS NOT NULL
GROUP BY brand
ORDER BY average_price DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,brand,product_count,minimum_price,average_price,maximum_price
0,Kitcorner,1,751.0,751.0,751.0
1,Loknath,1,567.0,567.0,567.0
2,GRCOXZ,1,560.0,560.0,560.0
3,Veksin,1,497.0,497.0,497.0
4,Hoatzin,1,497.0,497.0,497.0
5,Sequence Products,1,489.0,489.0,489.0
6,SKYHEART,1,486.0,486.0,486.0
7,AneriDEALS,1,423.0,423.0,423.0
8,Q7 BRAND,1,363.0,363.0,363.0
9,RK,2,333.0,358.0,383.0


<div align="center">

###  7. Material-wise Analysis

**Analyzing product count and average price across different materials.**

</div>

In [8]:
query = """
SELECT 
    material,
    COUNT(*) AS product_count,
    ROUND(AVG(price), 2) AS average_price
FROM products
WHERE material IS NOT NULL
GROUP BY material
ORDER BY product_count DESC, average_price DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,material,product_count,average_price
0,Plastic,20,319.1
1,Steel,3,489.0
2,"Stainless Steel, Steel",2,227.5
3,"Plastic, Steel",1,567.0
4,Ceramic,1,345.0
5,"Stainless Steel, Plastic, Steel",1,292.0
6,Glass,1,169.0


<div align="center">

###  8. Discount-wise Product Analysis

**Analyzing product count, average rating, and average reviews across different discount levels.**

</div>

In [9]:
query = """
SELECT 
    discount,
    COUNT(*) AS product_count,
    ROUND(AVG(rating), 2) AS average_rating,
    ROUND(AVG(reviews), 2) AS average_reviews
FROM products
WHERE discount IS NOT NULL
GROUP BY discount
ORDER BY discount
"""

display(pd.read_sql(query, query_conn))

,discount,product_count,average_rating,average_reviews
0,6,1,4.10,94.0
1,15,1,4.30,43.0
2,31,1,3.80,NaN
3,43,1,4.10,3.0
4,50,3,4.23,160.5
5,51,2,4.20,11.0
6,54,1,4.20,4.0
7,61,1,4.40,61.0
8,62,1,4.10,3.0
9,64,1,4.30,23.0


<div align="center">

###  9. Review Coverage Analysis

**Analyzing product review availability and calculating the percentage of products with customer reviews.**

</div>

In [10]:
query = """
SELECT 
    COUNT(*) AS total_products,
    SUM(CASE WHEN reviews IS NOT NULL THEN 1 ELSE 0 END) AS products_with_reviews,
    SUM(CASE WHEN reviews IS NULL THEN 1 ELSE 0 END) AS products_without_reviews,
    ROUND(
        100 * AVG(
            CASE WHEN reviews IS NOT NULL THEN 1 ELSE 0 END
        ), 
        2
    ) AS review_coverage_percent
FROM products
"""

display(pd.read_sql(query, query_conn))

,total_products,products_with_reviews,products_without_reviews,review_coverage_percent
0,29,26,3,89.66


<div align="center">

###  10. Price Band Analysis

**Analyzing product count, average rating, and total reviews across different price ranges.**

</div>

In [11]:
query = """
SELECT 
    CASE 
        WHEN price < 200 THEN 'Under 200'
        WHEN price < 400 THEN '200 to 399'
        ELSE '400 and above'
    END AS price_band,
    COUNT(*) AS product_count,
    ROUND(AVG(rating), 2) AS average_rating,
    COALESCE(SUM(reviews), 0) AS total_reviews
FROM products
WHERE price IS NOT NULL
GROUP BY price_band
ORDER BY MIN(price)
"""

display(pd.read_sql(query, query_conn))

,price_band,product_count,average_rating,total_reviews
0,Under 200,6,3.92,101.0
1,200 to 399,15,4.14,1088.0
2,400 and above,8,4.15,222.0


<div align="center">

###  11. Product Engagement Score

**Identifying highly rated products with strong customer engagement based on ratings and review counts.**

</div>

In [12]:
query = """
SELECT 
    product_name,
    brand,
    price,
    discount,
    rating,
    reviews,
    ROUND(
        rating * LOG10(COALESCE(reviews, 0) + 1),
        2
    ) AS engagement_score
FROM products
WHERE rating >= 4
  AND price IS NOT NULL
ORDER BY engagement_score DESC, discount DESC
LIMIT 10
"""

display(pd.read_sql(query, query_conn))

,product_name,brand,price,discount,rating,reviews,engagement_score
0,koktailkitchan Pack of 12 Plastic Grocery Cont...,koktailkitchan,322,50,4.1,314.0,10.24
1,BELIZZI Pack of 6 Plastic Fridge Container - 1...,BELIZZI,257,74,4.1,272.0,9.99
2,VR Pack of 3 Plastic Grocery Container - 4500 ...,VR,247,75,4.1,210.0,9.53
3,Qtrix Pack of 8 Plastic Grocery Container - 50...,Qtrix,300,69,4.4,104.0,8.89
4,GRCOXZ Pack of 12 Plastic Grocery Container - ...,GRCOXZ,560,6,4.1,94.0,8.11
5,"RK Pack of 5 Steel Cookie Jar - 300 ml, 500 ml...",RK,383,76,4.1,83.0,7.89
6,SKYHEART Pack of 8 Plastic Grocery Container -...,SKYHEART,486,61,4.4,61.0,7.89
7,VR Pack of 4 Plastic Fridge Container - 2500 m...,VR,292,70,4.1,70.0,7.59
8,VAD Pack of 2 Glass Grocery Container - 1000 m...,VAD,169,15,4.3,43.0,7.07
9,AneriDEALS Pack of 24 Plastic Grocery Containe...,AneriDEALS,423,78,4.1,34.0,6.33


<div align="center">

### Conclusion

**SQL analysis provided useful insights into product ratings, reviews, discounts, pricing, brands, materials, and customer engagement.**

**The analysis helps identify top-performing products and supports data-driven business decisions.**

</div>